In [1]:
import tomllib
from pathlib import Path

import pandas as pd


In [2]:
file_path = Path("/home/japlan001/PycharmProjects/paper/data/input/datahub")
config_path = Path("/home/japlan001/PycharmProjects/paper/configs/datahub.toml")
output_path = Path("./filtered_datahub")

# ensure that the outputpath exists
output_path.mkdir(parents=True, exist_ok=True)

In [3]:
config = tomllib.loads(config_path.read_text(encoding="utf-8"))

In [4]:
def append_single(append_list: list, item: any) -> None:
    """Append all single items from item to create a flat list."""
    if isinstance(item, list):
        for subitem in item:
            append_single(append_list, subitem)
    elif isinstance(item, dict):
        for value in item.values():
            append_single(append_list, value)
    else:
        append_list.append(item)

In [5]:
def extract_requested_headers(config: dict) -> list:
    # we need these two for the plugin to work
    requested_headers = ["sample:analysis-pid", "sample:pid"]
    keys = ["combined_columns", "split_fields", "mapping"]

    for key in keys:
        for item in pd.json_normalize(config.get(key), sep=".").values.flatten():
            append_single(requested_headers, item)

    # only keep
    requested_headers = [header for header in requested_headers if ":" in str(header) and not str(header).startswith("Literal:")]
    return requested_headers

In [6]:
requested_headers = extract_requested_headers(config)

In [7]:
for file in file_path.glob("*.xlsx"):
    sheets = pd.read_excel(file, header=0, skiprows=[1, 2], sheet_name=None)

    # ignore all sheets, where not at least one of the following is provided
    dataset = sheets["dataset"]
    pid_cols = ["dataset:pid", "dataset:pid-source", "dataset:pid-url"]
    if not dataset[pid_cols].notna().values.any():
        continue

    for sheet, df in sheets.items():
        sheets[sheet] = df.loc[:, df.columns.isin(requested_headers)]

    with pd.ExcelWriter(output_path / file.name) as writer:
        for sheet, df in sheets.items():
            df.to_excel(writer, sheet_name=sheet, index=False)

In [8]:
requested_headers

['sample:analysis-pid',
 'sample:pid',
 'author:first-name',
 'author:last-name',
 'analysis:author-pid',
 'analysis:sample',
 'analysis:keywords',
 'analysis:result',
 'dataset:keywords',
 'file:sample',
 'author:first-name',
 'author:last-name',
 'author:pid',
 'author:affiliation',
 'dataset:title',
 'dataset:description',
 'dataset:pid',
 'dataset:pid-url',
 'dataset:pid-source',
 'dataset:date-first-created',
 'dataset:date-last-updated',
 'dataset:date-public',
 'dataset:language',
 'dataset:license',
 'dataset:keywords',
 'dataset:b5d-work-package',
 'author:is-dataset-author',
 'analysis:title',
 'analysis:pid',
 'analysis:pid-url',
 'analysis:pid-source',
 'analysis:description',
 'analysis:author-pid',
 'analysis:sample',
 'analysis:method',
 'analysis:result',
 'analysis:category',
 'analysis:keywords',
 'file:identifier',
 'file:name',
 'file:url',
 'file:format',
 'file:description',
 'file:comment',
 'file:sample']